# Flash-flood impacts: where the losses concentrate

NOAA NCEI / NWS · 2020–2024 · 50 states and Washington, D.C.

This notebook reviews the built analytical model. Cleaning and model construction are implemented once in `src/floodfolio/`; run `python run_pipeline.py` before executing the notebook. All saved code outputs were executed against the frozen source snapshot.

The business question is where an operations review should start when recorded event volume, financial impact, and human impact produce different priorities.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
connection = sqlite3.connect(ROOT / 'data/processed/flash_flood.sqlite')

def query(filename):
    return pd.read_sql_query((ROOT / 'sql' / filename).read_text(), connection)

for table in ['fact_event', 'fact_location', 'fact_fatality', 'dim_episode', 'dim_geography']:
    count = connection.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:20s} {count:>8,}')


fact_event             19,191
fact_location          58,578
fact_fatality             401
dim_episode             5,162
dim_geography           2,383


## 1. Reconcile the source grain before calculating impact

Locations and fatalities are separate one-to-many tables. Counting unique event IDs after a direct join would not repair duplicated monetary sums.

In [2]:
correct = connection.execute('SELECT SUM(property_damage_usd) FROM fact_event').fetchone()[0]
naive = connection.execute('''
    SELECT COUNT(*), SUM(e.property_damage_usd)
    FROM fact_event e
    LEFT JOIN fact_location l USING(event_id)
    LEFT JOIN fact_fatality f USING(event_id)
''').fetchone()
protected = connection.execute('SELECT COUNT(*), SUM(property_damage_usd) FROM vw_event_enriched').fetchone()
print(f'Protected event rows: {protected[0]:,}')
print(f'Protected damage: ${protected[1]:,.0f}')
print(f'Naive joined rows: {naive[0]:,}')
print(f'Naive joined damage: ${naive[1]:,.0f}')
print(f'Inflation factor: {naive[1] / correct:.4f}x')
assert protected[1] == correct


Protected event rows: 19,191
Protected damage: $8,780,404,020
Naive joined rows: 64,795
Naive joined damage: $43,174,952,550
Inflation factor: 4.9172x


## 2. Impact scorecard

Event-summary counts supply the casualty KPIs. Property and crop losses remain separate; all monetary values are nominal reported estimates.

In [3]:
headline = query('01_headline_kpis.sql')
print(headline.T.to_string(header=False))


recorded_events                    1.919100e+04
recorded_episodes                  5.162000e+03
counties_with_records              2.383000e+03
reported_property_damage_usd       8.780404e+09
reported_crop_damage_usd           2.334270e+08
reported_direct_deaths             3.890000e+02
reported_indirect_deaths           1.000000e+01
reported_direct_injuries           9.300000e+01
reported_indirect_injuries         1.400000e+01
positive_property_events           5.713000e+03
zero_property_events               1.347800e+04
unknown_property_events            0.000000e+00
direct_fatal_events                1.980000e+02
mean_recorded_duration_hours       2.647896e+00
mean_positive_property_damage_usd  1.536917e+06


## 3. Concentration, not the average event

The top-1% denominator is all event records, including explicit zero-loss entries. The selected count is rounded up, with event ID breaking ties deterministically.

In [4]:
print(query('04_loss_concentration.sql').to_string(index=False))
events = pd.read_sql_query('SELECT * FROM vw_event_enriched', connection)
positive = events.loc[events.property_damage_usd > 0, 'property_damage_usd']
print(f'Positive-loss median: ${positive.median():,.0f}')
print(f'Positive-loss mean: ${positive.mean():,.0f}')


      population  selected_records  selected_loss_usd  all_loss_usd  loss_share_pct
Top 1% of events               192       8362026000.0  8780404020.0       95.235094
   Top 10 events                10       3850000000.0  8780404020.0       43.847641
Positive-loss median: $10,000
Positive-loss mean: $1,536,917


![Loss concentration](../reports/figures/02_loss_concentration.png)

A default IQR filter on the untransformed, zero-heavy loss distribution would be a poor choice. The sensitivity analysis below preserves the complete impact population.

## 4. Event frequency and financial impact

The monthly SQL uses a calendar spine so zero-event months cannot disappear. All 60 monthly observations are retained in the chart and exported table.

In [5]:
monthly = query('02_monthly_trend.sql')
print(monthly.groupby('year')[['events', 'property_damage_usd', 'direct_deaths']].sum().to_string())
print(f'Months in calendar spine: {len(monthly)}')
print(query('03_state_comparison.sql')[['state','events','property_damage_usd','direct_deaths']].head(8).to_string(index=False))


      events  property_damage_usd  direct_deaths
year                                            
2020    3485         4.573655e+08             37
2021    4748         2.423851e+09            134
2022    2981         1.165906e+09             70
2023    3518         1.591061e+09             37
2024    4459         3.142221e+09            111
Months in calendar spine: 60
         state  events  property_damage_usd  direct_deaths
NORTH CAROLINA     966         2162857000.0             82
         TEXAS    1504          955459200.0             15
    CALIFORNIA     626          725633000.0             13
     LOUISIANA     638          633769000.0              7
      NEW YORK     991          606931000.0             18
      ILLINOIS     441          594234000.0              3
    NEW MEXICO     347          539000400.0             14
  PENNSYLVANIA     846          377506500.0             16


![Monthly impacts](../reports/figures/04_monthly_impacts.png)

## 5. Overnight-start profile

Overnight means 20:00–05:59 in NOAA's reported local time. It describes event start, not the time of death. The comparison is unadjusted and descriptive.

In [6]:
overnight = query('05_overnight_start.sql')
print(overnight.to_string(index=False))
ratio = overnight.set_index('start_period').fatal_event_pct
print(f'Unadjusted fatal-event proportion ratio: {ratio["Overnight start"] / ratio["Other start"]:.3f}x')


   start_period  events  direct_deaths  direct_fatal_events  fatal_event_pct  direct_deaths_per_1000_recorded_events  property_damage_usd
Overnight start    5528            206                   99         1.790883                               37.264834         3246431690.0
    Other start   13663            183                   99         0.724585                               13.393837         5533972330.0
Unadjusted fatal-event proportion ratio: 2.472x


![Overnight profile](../reports/figures/03_overnight_profile.png)

## 6. What the source tables disagree about

The mismatch is retained. The event-summary total of 389 direct deaths and the 391 direct-fatality child records answer different source-specific questions.

In [7]:
print(query('07_fatality_reconciliation.sql').to_string(index=False))
print(query('06_fatality_context.sql').to_string(index=False))


 event_id  year   state    cz_name  deaths_direct  direct_fatality_records  direct_difference  deaths_indirect  indirect_fatality_records  indirect_difference         source_file  source_row
   970467  2021 ALABAMA TUSCALOOSA              0                        2                  2                0                          0                    0 details_2021.csv.gz         222
    fatality_location  direct_fatality_records  unknown_age_records  share_all_direct_records_pct  share_known_location_direct_records_pct
Vehicle/Towed Trailer                      116                   10                     29.667519                                34.834835
       Permanent Home                       77                    3                     19.693095                                23.123123
             In Water                       60                    3                     15.345269                                18.018018
   Outside/Open Areas                       51                

## 7. Reporting quality beside the impact totals

A zero monetary entry is not a missing field. Child-location coverage uses distinct parent events, not point counts.

In [8]:
print(pd.read_csv(ROOT / 'reports/tables/quality_by_year.csv').to_string(index=False))
casualty = events.deaths_direct.add(events.injuries_direct).gt(0)
zero = events.property_damage_status.eq('zero')
print(f'Direct-casualty events: {casualty.sum():,}')
print(f'Direct-casualty events with zero property-damage entry: {(casualty & zero).sum():,}')
print(f'Among casualty events: {100 * zero[casualty].mean():.2f}%')


 year  events  with_locations  zero_damage  location_coverage_pct  zero_damage_pct
 2020    3485            3416         2454              98.020086        70.416069
 2021    4748            3715         3317              78.243471        69.860994
 2022    2981            1786         1901              59.912781        63.770547
 2023    3518            2024         2655              57.532689        75.469016
 2024    4459            2542         3151              57.008298        70.666069
Direct-casualty events: 218
Direct-casualty events with zero property-damage entry: 101
Among casualty events: 46.33%


![Reporting coverage](../reports/figures/05_reporting_coverage.png)

## 8. Outlier removal changes the analytical question

Duration flags use an upper 1.5×IQR fence. Property-loss flags apply that fence to log(1 + positive loss). The full population supplies headline impacts; the routine population excludes flags for comparison only.

In [9]:
print(query('09_outlier_sensitivity.sql').to_string(index=False))
print(events[['duration_hours', 'property_damage_usd']].quantile([.5, .9, .95, .99]).to_string())
print('Long-duration records awaiting source review:', int(events.duration_hours.gt(72).sum()))
connection.close()


                   population  events  property_damage_usd  mean_property_damage_usd  direct_deaths  mean_duration_hours
              Full population   19191         8780404020.0             457527.175238            389             2.647896
    Duration outliers removed   18105         3544687870.0             195785.024579            246             2.091520
      Damage outliers removed   19031          507754020.0              26680.364668            252             2.602101
Routine comparison population   18012          422277870.0              23444.252165            203             2.084271
      duration_hours  property_damage_usd
0.50        2.000000                  0.0
0.90        5.100000              25000.0
0.95        6.366667             100000.0
0.99       13.000000            2000000.0
Long-duration records awaiting source review: 9


![Outlier sensitivity](../reports/figures/06_outlier_sensitivity.png)

## Implications

Separate high-impact episode review from routine event activity. Add reporting diagnostics to both. Investigate the overnight association only after adding severity, geography, and exposure information. Resolve the casualty-table discrepancy through source review.

The evidence does not establish causal effects, population-normalized risk, insurance losses, warning effectiveness, or a measured operational improvement. See the decision memo and limitations document for the proposed next stage.